In [ ]:
# 
#invite people for the Kaggle party
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
# STEFANOS: Unneeded
# from scipy.stats import norm
from sklearn.preprocessing import StandardScaler
# from scipy import stats
import warnings
import time
warnings.filterwarnings('ignore')
# STEFANOS: Disable magic functions
# %matplotlib inline

In [ ]:
%%time
### cell 0 ###

#bring in the six packs
df_train = pd.read_csv('/home/dias-benchmarks/notebooks/pmarcelino/comprehensive-data-exploration-with-python/input/train.csv')
factor = 300
df_train = pd.concat([df_train]*factor, ignore_index=True)
df_train.info()

In [ ]:
%%time
### cell 1 ###

#check the decoration
df_train.columns

In [ ]:
%%time
### cell 2 ###

#descriptive statistics summary
df_train['SalePrice'].describe()

In [ ]:
%%time
### cell 3 ###

#skewness and kurtosis
print("Skewness: %f" % df_train['SalePrice'].skew())
print("Kurtosis: %f" % df_train['SalePrice'].kurt())

In [ ]:
%%time
### cell 4 ###

#scatter plot grlivarea/saleprice
var = 'GrLivArea'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)

In [ ]:
%%time
### cell 5 ###

#scatter plot totalbsmtsf/saleprice
var = 'TotalBsmtSF'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)

In [ ]:
%%time
### cell 6 ###

#box plot overallqual/saleprice
var = 'OverallQual'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)

In [ ]:
%%time
### cell 7 ###

var = 'YearBuilt'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)

In [ ]:
%%time
### cell 8 ###

numeric_df = df_train.select_dtypes(include=["number"])
corrmat = numeric_df.corr()

In [ ]:
%%time
### cell 9 ###

#saleprice correlation matrix
k = 10 #number of variables for heatmap
cols = corrmat.sort_values(by='SalePrice', ascending=False).head(k)['SalePrice'].index
cm = np.corrcoef(df_train[cols].values.T)

In [ ]:
%%time
### cell 10 ###

#missing data
total = df_train.isnull().sum().sort_values(ascending=False)
percent = (df_train.isnull().sum()/df_train.isnull().count()).sort_values(ascending=False)
missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
missing_data.head(20)

In [ ]:
%%time
### cell 11 ###

#dealing with missing data
df_train = df_train.drop(missing_data[missing_data['Total'] > 1].index, axis=1)
df_train.isnull().sum().max() #just checking that there's no missing data missing...

In [ ]:
%%time
### cell 12 ###

#standardizing data
saleprice_scaled = StandardScaler().fit_transform(df_train['SalePrice'].to_numpy()[:, np.newaxis])
low_range = saleprice_scaled[saleprice_scaled[:,0].argsort()][:10]
high_range= saleprice_scaled[saleprice_scaled[:,0].argsort()][-10:]
print('outer range (low) of the distribution:')
print(low_range)
print('\nouter range (high) of the distribution:')
print(high_range)

In [ ]:
%%time
### cell 13 ###

#bivariate analysis saleprice/grlivarea
var = 'GrLivArea'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)

In [ ]:
%%time
### cell 14 ###

# deleting points
# Display the two houses with largest living area more efficiently
df_train.nlargest(2, 'GrLivArea')

# Remove both outliers in one vectorized step
ids_to_remove = [1299, 524]
df_train = df_train.loc[~df_train['Id'].isin(ids_to_remove)]

In [ ]:
%%time
### cell 15 ###

#bivariate analysis saleprice/grlivarea
var = 'TotalBsmtSF'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)

In [ ]:
%%time
### cell 16 ###

#applying log transformation
df_train['SalePrice'] = np.log(df_train['SalePrice'])

In [ ]:
%%time
### cell 17 ###

#data transformation
df_train['GrLivArea'] = np.log(df_train['GrLivArea'])

In [ ]:
%%time
### cell 18 ###

df_train['HasBsmt'] = (df_train['TotalBsmtSF'] > 0).astype(int)

In [ ]:
%%time
### cell 19 ###

# Compute mask and update the underlying numpy array in place
mask = df_train['HasBsmt'].values == 1
bsmt_arr = df_train['TotalBsmtSF'].values
bsmt_arr[mask] = np.log(bsmt_arr[mask])

In [ ]:
%%time
### cell 20 ###

#histogram and normal probability plot
_ = df_train[df_train['TotalBsmtSF']>0]['TotalBsmtSF']
_ = df_train[df_train['TotalBsmtSF']>0]['TotalBsmtSF']

In [ ]:
%%time
### cell 21 ###

#scatter plot
_ = df_train[df_train['TotalBsmtSF']>0]['TotalBsmtSF']
_ = df_train[df_train['TotalBsmtSF']>0]['SalePrice']

In [ ]:
%%time
### cell 22 ###

#convert categorical variable into dummy
df_train = pd.get_dummies(df_train)